<p>
  <img style="display: block; margin-left: auto; margin-right: auto; border-radius: 12px;" src="https://tse3.mm.bing.net/th/id/OIP.ELWM8dJab3LmOkwzMgH7EwHaHa?rs=1&pid=ImgDetMain&o=7&rm=3" alt="" width="140" height="140" />
</p>

<h1 style="text-align: center;">
  <span style="color: #00ffff;">🎮 Servidor de Minecraft en Colab — CloudCraft</span>
</h1>
<hr />

<div style="background: linear-gradient(135deg, #1e293b, #0f172a); border: 2px solid #10b981; border-radius: 12px; padding: 20px; text-align: center; color: #f8fafc; font-family: sans-serif;">
  <h3 style="color: #10b981; margin-top: 0;">🚀 ¿COMO ENCENDER EL SERVIDOR?</h3>
  <p style="font-size: 15px; margin-bottom: 12px;">
    Para encender el servidor y jugar con tus amigos, haz clic arriba en el menú:<br>
    <strong style="color: #38bdf8; font-size: 16px;">Entorno de ejecución ➔ Ejecutar todo</strong> (o presiona <code style="background: #334155; padding: 2px 8px; border-radius: 4px;">Ctrl + F9</code>)
  </p>
  <span style="font-size: 12px; color: #94a3b8;">Toda la configuración, mundos y tu IP de Playit.gg se cargan automáticamente.</span>
</div>
<hr />


----


----
# &#128640; **Iniciar la maquina**
---
Esta sección te permite encender la máquina virtual en Google Colab.

In [ ]:
# @title ## **[⚙] Configuración Inicial (Set up)**
# @markdown Inicializa las librerías necesarias y monta Google Drive.
import subprocess, sys, os

def pip_silent(pkg, import_name=None):
    name = import_name or pkg
    try:
        __import__(name)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg,
                               '--progress-bar', 'off'])

pip_silent('requests')
pip_silent('flask', 'flask')
pip_silent('psutil')
pip_silent('bs4', 'bs4')
pip_silent('mcstatus')
pip_silent('pyngrok')
pip_silent('rich')
pip_silent('ruamel.yaml', 'ruamel')

import requests, json, concurrent.futures
from time import sleep
from os.path import exists
from os import makedirs
from IPython.display import clear_output
from rich import print

print("[bold green]✅ Librerías cargadas correctamente.[/bold green]")

# ── Montar Google Drive con reintentos ──────────────────────────────────────
def mount_drive(max_retries=3):
    if os.path.ismount('/content/drive'):
        print("[bold blue]ℹ Google Drive ya está montado.[/bold blue]")
        return True
    from google.colab import drive
    for attempt in range(1, max_retries + 1):
        try:
            print(f"[bold yellow]Intento {attempt} de montar Google Drive...[/bold yellow]")
            drive.mount('/content/drive', force_remount=(attempt > 1))
            if os.path.ismount('/content/drive'):
                print("[bold green]✅ Google Drive montado correctamente.[/bold green]")
                return True
        except Exception as e:
            print(f"[bold red]⚠ Intento {attempt} fallido: {e}[/bold red]")
            if attempt < max_retries:
                print("[yellow]Esperando 5 segundos antes del siguiente intento...[/yellow]")
                sleep(5)
    print("[bold red]❌ No se pudo montar Google Drive. Verifica tu conexión y autorización.[/bold red]")
    return False

mount_ok = mount_drive()

drive_path = '/content/drive/MyDrive/minecraft'
SERVERCONFIG = f'{drive_path}/server_list.txt'

if mount_ok:
    makedirs(drive_path, exist_ok=True)
    if not exists(SERVERCONFIG):
        json.dump({"server_list": [], "server_in_use": "",
                   "ngrok_proxy": {"authtoken": "", "region": "us"},
                   "zrok_proxy": {"authtoken": ""},
                   "playit_proxy": {"secretkey": ""},
                   "localtonet_proxy": {"authtoken": ""}},
                  open(SERVERCONFIG, 'w'))

# ── Información de la VM ────────────────────────────────────────────────────
colabversion = "0.4.0"
try:
    def fetch_json(url):
        try:
            return requests.get(url, timeout=5).json()
        except:
            return {}

    with concurrent.futures.ThreadPoolExecutor() as executor:
        future_ip = executor.submit(fetch_json, "https://ipinfo.io/")
        ipinfo = future_ip.result() or {}

    if ipinfo:
        ip   = ipinfo.get('ip',     'N/A')
        city = ipinfo.get('city',   'N/A')
        reg  = ipinfo.get('region', 'N/A')
        ctr  = ipinfo.get('country','N/A')
        print(f"\n[bold cyan]VM Info — IP: {ip} | {city}, {reg}, {ctr}[/bold cyan]")
except Exception as e:
    print(f"[yellow]No se pudo obtener info de VM: {e}[/yellow]")

print(f"[bold green]✅ CloudCraft v{colabversion} — Setup completado.[/bold green]")


----
# 🚀 **Panel de Control Web (Dashboard)**
---
Interfaz interactiva de **CloudCraft** para gestionar tu servidor de Minecraft desde el navegador.


In [ ]:
# @title ## **[⚡] Iniciar Panel de Control Web & Anti-Desconexión**
# @markdown Ejecuta esta celda para iniciar el panel web y mantener la sesión de Colab activa.
import os, time, json, base64, subprocess, sys, re, glob
from IPython.display import clear_output, display, HTML

def pip_silent(pkg, import_name=None):
    name = import_name or pkg
    try:
        __import__(name)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg,
                               '--progress-bar', 'off'])

pip_silent('flask', 'flask')
pip_silent('psutil')
pip_silent('requests')
pip_silent('bs4', 'bs4')
pip_silent('mcstatus')

# Detección Inteligente de Carpeta de Drive (Propia o Compartida)
possible_paths = [
    '/content/drive/MyDrive/minecraft',
    '/content/drive/MyDrive/Shared with me/minecraft',
    '/content/drive/MyDrive/Compartido conmigo/minecraft'
]
drive_path = None
for p in possible_paths:
    if os.path.exists(p):
        drive_path = p
        break

if not drive_path:
    shortcuts = glob.glob('/content/drive/MyDrive/.shortcut-targets-by-id/*/minecraft')
    if shortcuts:
        drive_path = shortcuts[0]

if not drive_path:
    sdrives = glob.glob('/content/drive/Shareddrives/*/minecraft')
    if sdrives:
        drive_path = sdrives[0]

if not drive_path:
    drive_path = '/content/drive/MyDrive/minecraft'
    os.makedirs(drive_path, exist_ok=True)

print(f"📁 Carpeta de Minecraft conectada: {drive_path}")

print("Desplegando archivos del panel web...")
dashboard_b64 = '...'
colab_panel_b64 = '...'

with open(os.path.join(drive_path, 'dashboard.html'), 'wb') as f:
    f.write(base64.b64decode(dashboard_b64.encode('utf-8')))

with open(os.path.join(drive_path, 'colab_panel.py'), 'wb') as f:
    f.write(base64.b64decode(colab_panel_b64.encode('utf-8')))

print("Archivos escritos correctamente.")

os.system('pkill -f colab_panel.py 2>/dev/null || true')
time.sleep(1)

print("Iniciando servidor backend en puerto 8000...")
flask_proc = subprocess.Popen(
    [sys.executable, os.path.join(drive_path, 'colab_panel.py')],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
)
time.sleep(4)

# Generar Túnel Público HTTPS para acceder al panel desde cualquier navegador
cf_url = "Iniciando túnel web..."
try:
    if not os.path.exists('/tmp/cloudflared'):
        subprocess.run(['wget', '-q', 'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64', '-O', '/tmp/cloudflared'], check=False)
        subprocess.run(['chmod', '+x', '/tmp/cloudflared'], check=False)
    
    cf_proc = subprocess.Popen(['/tmp/cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8000'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for _ in range(30):
        line = cf_proc.stdout.readline()
        if not line:
            break
        m = re.search(r'https://[a-zA-Z0-9-]+\x2etrycloudflare\x2ecom', line)
        if not m:
            m = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
        if m:
            cf_url = m.group(0)
            break
        time.sleep(0.2)
except Exception:
    cf_url = "https://127.0.0.1:8000"

from google.colab.output import eval_js
try:
    tunnel_link = eval_js("google.colab.kernel.proxyPort(8000)")
except Exception:
    tunnel_link = cf_url

clear_output()

print("=" * 65)
print("🚀 PANEL CLOUDCRAFT LISTO")
print("=" * 65)
print(f"📁 CARPETA CONECTADA: {drive_path}")
print(f"🌐 ENLACE PUBLICO DEL PANEL: {cf_url}")
print("=" * 65)

html_box = f"""
<div style="border: 2px solid #10b981; border-radius: 14px; padding: 24px;
            background: linear-gradient(135deg,#0b0f19,#141d30);
            color: #f3f4f6; font-family: 'Segoe UI',sans-serif;
            max-width: 640px; margin: 20px auto; text-align: center;
            box-shadow: 0 10px 30px rgba(0,0,0,0.6);">
  <h2 style="color:#10b981; margin-top:0; font-size:22px;">🚀 Panel CloudCraft Listo</h2>
  <p style="color:#9ca3af; margin-bottom:12px; font-size:14px;">
    Accede al panel de control de CloudCraft desde el siguiente enlace:
  </p>
  <a href="{tunnel_link}" target="_blank"
     style="display:inline-block; background:linear-gradient(135deg,#10b981,#059669);
            color:#0b0f19; font-weight:700; text-decoration:none;
            padding:14px 32px; border-radius:8px; font-size:16px;
            box-shadow:0 4px 15px rgba(16,185,129,0.4); margin-bottom:16px;">
    Abrir Panel de Control
  </a>
  
  <div style="background: rgba(56, 189, 248, 0.12); border: 1px solid rgba(56, 189, 248, 0.35); border-radius: 10px; padding: 12px; margin-top: 10px; text-align: center;">
    <strong style="color: #38bdf8; font-size: 13px;">🌐 Enlace Público del Panel (Para compartir con amigos):</strong><br>
    <div style="margin-top: 6px;">
      <code style="color: #4ade80; font-family: monospace; font-size: 14px; background: rgba(0,0,0,0.3); padding: 4px 10px; border-radius: 6px;">{cf_url}</code>
    </div>
  </div>
</div>

<script>
  function keepColabAlive() {
    try {
      const btn = document.querySelector("colab-connect-button");
      if (btn) btn.click();
    } catch(e) {}
  }
  setInterval(keepColabAlive, 60000);
</script>
"""
display(HTML(html_box))

try:
    while True:
        time.sleep(10)
        if flask_proc.poll() is not None:
            print("⚠ El backend se detuvo inesperadamente. Reiniciando...")
            flask_proc = subprocess.Popen(
                [sys.executable, os.path.join(drive_path, 'colab_panel.py')],
                stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
            )
            time.sleep(3)
except KeyboardInterrupt:
    print("Deteniendo panel web...")
    flask_proc.terminate()
